# 03 · Evaluación: baseline, adapters y experimento de inferencia

Calcula la **EX ponderada** sobre la validación propia para:
1. Baseline (modelo base, sin adapter).
2. Adapter A y Adapter B.
3. El mejor adapter **con** auto-corrección vía `ejecutar` (experimento de inferencia).

Luego hace el análisis de errores por categoría.

In [ ]:
# Clonar el repositorio en Colab (ajusta la URL a tu repo privado)
# !git clone https://github.com/USUARIO/proyecto-dl-3.git
# %cd proyecto-dl-3
import sys, os
# Si ejecutas localmente, asegúrate de estar en la raíz del repo.
assert os.path.exists("src"), "Ejecuta desde la raíz del repositorio."


In [ ]:
!pip -q install -r requirements.txt pymysql


## Función auxiliar: evaluar una configuración
Reescribe `config.json` (modelo_base/adapter/usar_adapter), reimporta `solucion.py` y corre el harness.

In [ ]:
import importlib, json, sys
from src.evaluar import evaluar_validacion, guardar_detalle

def eval_config(usar_adapter, adapter='adapter/', usar_ejecutar=False):
    cfg=json.load(open('config.json'))
    cfg['usar_adapter']=usar_adapter; cfg['adapter']=adapter
    json.dump(cfg, open('config.json','w'), ensure_ascii=False, indent=2)
    for m in list(sys.modules):
        if m=='solucion' or m.startswith('solucion'): del sys.modules[m]
    import solucion
    ex, det = evaluar_validacion(solucion.generar_sql, usar_ejecutar=usar_ejecutar)
    return ex, det, solucion


## 1. Baseline (sin adapter)

In [ ]:
ex_base, det_base, _ = eval_config(usar_adapter=False)
print('EX baseline:', round(ex_base,4))
guardar_detalle(det_base, 'resultados/detalle_baseline.csv')


## 2. Adapter A y Adapter B

In [ ]:
ex_A, det_A, _ = eval_config(True, 'adapter_A')
ex_B, det_B, _ = eval_config(True, 'adapter_B')
print('EX adapter A (r=8):', round(ex_A,4))
print('EX adapter B (r=32):', round(ex_B,4))


## 3. Mejor adapter + experimento de inferencia (auto-corrección)

In [ ]:
mejor = 'adapter_B' if ex_B>=ex_A else 'adapter_A'
ex_corr, det_corr, _ = eval_config(True, mejor, usar_ejecutar=True)
print('Mejor adapter:', mejor)
print('EX con auto-corrección:', round(ex_corr,4))
guardar_detalle(det_corr, 'resultados/detalle_mejor_autocorreccion.csv')


## 4. Tabla resumen y mejora sobre baseline

In [ ]:
import csv
filas=[('baseline',ex_base),('adapter_A r=8',ex_A),('adapter_B r=32',ex_B),('mejor+autocorr',ex_corr)]
with open('resultados/validacion_propia.csv','w',newline='',encoding='utf-8') as f:
    w=csv.writer(f); w.writerow(['configuracion','EX'])
    for n,e in filas: w.writerow([n, round(e,4)])
print('Mejora mejor-adapter sobre baseline:', round(max(ex_A,ex_B)-ex_base,4))
for n,e in filas: print(f'{n:22s} EX={e:.4f}')


## 5. Análisis de errores por categoría

In [ ]:
from collections import Counter
det = det_corr  # o el que quieras analizar
errores=[d for d in det if not d['acierto']]
print('Errores:', len(errores),'/',len(det))
print('Por categoría:', dict(Counter(d['categoria_error'] for d in errores)))
print('Por nivel:', dict(Counter(d['nivel'] for d in errores)))
for d in errores[:8]:
    print('\n[',d['nivel'],d['categoria_error'],']',d['pregunta'])
    print('  REF :', d['sql_referencia'])
    print('  PRED:', d['pred_sql'])
